In [0]:
%run "/Workspace/Users/soumya.saha221@gmail.com/traffic_data_project/4. Common config"

In [0]:
dbutils.widgets.text(name='env',defaultValue='',label='Enter the Environment')
env = dbutils.widgets.get('env')

####Reading raw roads bronze table

In [0]:
def read_bronze_table(environment):
    print('Reading raw roads bronze table: ',end='')
    df = spark.readStream.table(f"{environment}catalog.bronze.raw_roads")
    print('Success!')
    return df

####Transforming roads data

In [0]:
def roads_transform(df):
    from pyspark.sql.functions import when,col
    print('Roads transformation started: ', end='')
    df_road_category = df.withColumn('Road_category_name', when(col('Road_category') == 'TA','Class A trunk road')
                       .when(col('Road_category') == 'TM','Class A trunk motor')
                       .when(col('Road_category') == 'PA','Class A principal road')
                       .when(col('Road_category') == 'PM','Class A principal motorway')
                       .when(col('Road_category') == 'M','Class B road')
                       .otherwise('NA'))
    print('Success!')
    return df_road_category

In [0]:
def roads_contains(df):
    from pyspark.sql.functions import col, when
    print('Roads type transformation started: ', end='')
    df_contains = df.withColumn('Road_type', when(col('Road_category_name').contains('Class A'),'Major Road')
                                .when(col('Road_category_name').contains('Class B'),'Minor Road')
                                .otherwise('NA'))
    print('Success!')
    return df_contains

####Writing to silver roads table

In [0]:
def write_to_silver(df, environment):
    print('Writing to silver roads table: ', end='')
    df.writeStream.format('delta')\
    .outputMode('append')\
    .option('checkpointLocation', f"{checkpoint}/silverRoadsLoad/checkpnt")\
      .trigger(availableNow = True) \
    .toTable(f"{environment}catalog.silver.silver_roads").awaitTermination()
    print('Success!!!')

####Calling all functions

In [0]:
#reading the raw roads bronze table
dfroads = read_bronze_table(env)

#handling null values
dfnonulls = handle_nulls(dfroads)

#handling duplicates
dfclean = drop_dup(dfnonulls)

#roads transformation
dfroads_transformed = roads_transform(dfclean)
df_final = roads_contains(dfroads_transformed)

#writing to silver roads table
write_to_silver(df_final, env)